#Set up

In [44]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import plotly.express as px
import textwrap

# Função do gráfico

In [55]:
# =========================================================
# 1) Quebra de texto para melhorar exibição dos itens
# =========================================================
def quebrar_texto_html(texto, largura=22):
    """
    Quebra um texto longo em múltiplas linhas usando <br>.
    """
    if pd.isna(texto):
        return texto

    texto = str(texto)
    return "<br>".join(
        textwrap.wrap(texto, width=largura, break_long_words=False)
    )


# =========================================================
# 2) Preparar labels quebrados dos itens
# =========================================================
def preparar_labels_quebrados(
    df,
    coluna_texto,
    nova_coluna=None,
    largura_quebra=22
):
    """
    Cria uma nova coluna com quebra de linha em HTML.
    """
    df = df.copy()

    if nova_coluna is None:
        nova_coluna = f"{coluna_texto}_label"

    df[nova_coluna] = df[coluna_texto].apply(
        lambda x: quebrar_texto_html(x, largura=largura_quebra)
    )

    return df


# =========================================================
# 3) Calcular métricas do grupo
# =========================================================
def calcular_metricas_grupo(
    df,
    coluna_grupo,
    coluna_valor
):
    """
    Calcula total por grupo e percentual do grupo no total geral.
    """
    df_grupo = (
        df.groupby(coluna_grupo, as_index=False)[coluna_valor]
        .sum()
        .rename(columns={coluna_valor: "valor_grupo"})
    )

    total_geral = df_grupo["valor_grupo"].sum()
    df_grupo["pct_total_grupo"] = df_grupo["valor_grupo"] / total_geral

    return df_grupo


# =========================================================
# 4) Abreviar nome do grupo, opcional
# =========================================================
def abreviar_texto(texto, limite=28):
    """
    Abrevia um texto longo com reticências.
    """
    texto = str(texto)
    if len(texto) <= limite:
        return texto
    return texto[: limite - 3] + "..."


# =========================================================
# 5) Montar label compacto do grupo
# =========================================================
def montar_label_grupo_compacto(
    df_grupo,
    coluna_grupo,
    coluna_valor_grupo="valor_grupo",
    coluna_pct="pct_total_grupo",
    nome_nova_coluna="label_grupo",
    casas_decimais_pct=1,
    mostrar_valor=True,
    mostrar_pct_total=True,
    abreviar_nome=True,
    limite_nome=28,
    separador=" | "
):
    """
    Monta um label compacto para o grupo em uma única linha.

    Exemplo:
    Doenças Ap. Circulatório | 766 | 25,5% do total
    """
    df_grupo = df_grupo.copy()

    nome = df_grupo[coluna_grupo].astype(str)

    if abreviar_nome:
        nome = nome.apply(lambda x: abreviar_texto(x, limite=limite_nome))

    partes = [nome]

    if mostrar_valor:
        partes.append(df_grupo[coluna_valor_grupo].round(0).astype(int).astype(str))

    if mostrar_pct_total:
        pct = (
            (df_grupo[coluna_pct] * 100)
            .round(casas_decimais_pct)
            .astype(str)
            + "%"
        )
        partes.append(pct)

    df_grupo[nome_nova_coluna] = partes[0]
    for parte in partes[1:]:
        df_grupo[nome_nova_coluna] = df_grupo[nome_nova_coluna] + separador + parte

    return df_grupo


# =========================================================
# 6) Montar label conjunto universo
# =========================================================
def montar_label_universo(
    df,
    coluna_valor,
    nome_nova_coluna="label_universo",
    texto_universo="Todos os atendimentos",
    casas_decimais_pct=1,
    mostrar_valor=True,
    mostrar_pct_total=True
):
    """
    Cria uma coluna constante com o rótulo do universo total.

    Exemplo:
    Todos os atendimentos | 3000 | 100,0%
    """
    df = df.copy()

    total = df[coluna_valor].sum()

    partes = [texto_universo]

    if mostrar_valor:
        partes.append(f"{int(round(total, 0))}")

    if mostrar_pct_total:
        partes.append(f"{100:.{casas_decimais_pct}f}%")

    label_universo = " | ".join(partes)
    df[nome_nova_coluna] = label_universo

    return df

# =========================================================
# 7) Preparar dataframe final
# =========================================================
def preparar_df_treemap(
    df,
    coluna_grupo,
    coluna_item,
    coluna_valor,
    largura_quebra_item=22,
    nome_coluna_label_item="item_label",
    nome_coluna_label_grupo="label_grupo",
    nome_coluna_label_universo="label_universo",
    abreviar_nome_grupo=True,
    limite_nome_grupo=28,
    incluir_universo=True,
    texto_universo="Todos os atendimentos"
):
    """
    Prepara dataframe final para o treemap.
    """
    df_plot = df.copy()

    df_plot = preparar_labels_quebrados(
        df=df_plot,
        coluna_texto=coluna_item,
        nova_coluna=nome_coluna_label_item,
        largura_quebra=largura_quebra_item
    )

    df_grupo = calcular_metricas_grupo(
        df=df_plot,
        coluna_grupo=coluna_grupo,
        coluna_valor=coluna_valor
    )

    df_grupo = montar_label_grupo_compacto(
        df_grupo=df_grupo,
        coluna_grupo=coluna_grupo,
        coluna_valor_grupo="valor_grupo",
        coluna_pct="pct_total_grupo",
        nome_nova_coluna=nome_coluna_label_grupo,
        abreviar_nome=abreviar_nome_grupo,
        limite_nome=limite_nome_grupo
    )

    df_plot = df_plot.merge(
        df_grupo[[coluna_grupo, "valor_grupo", "pct_total_grupo", nome_coluna_label_grupo]],
        on=coluna_grupo,
        how="left"
    )

    if incluir_universo:
        df_plot = montar_label_universo(
            df=df_plot,
            coluna_valor=coluna_valor,
            nome_nova_coluna=nome_coluna_label_universo,
            texto_universo=texto_universo
        )

    return df_plot

# =========================================================
# 8) Gráfico treemap agrupado
# =========================================================
def grafico_treemap_agrupado(
    df,
    coluna_grupo,
    coluna_item,
    coluna_valor,
    titulo="Treemap agrupado",
    subtitulo="Cada bloco mostra valor, % dentro do grupo e % do total.",
    cor=None,
    escala_cor="sunset",
    largura=1400,
    altura=800,
    largura_quebra_item=22,
    abreviar_nome_grupo=True,
    limite_nome_grupo=28,
    incluir_universo=True,
    texto_universo="Todos os atendimentos",
    mostrar_colorbar=True,
    titulo_colorbar=None,
    tamanho_fonte_texto=14,
    tamanho_fonte_geral=15,
    mostrar_valor_item=True,
    mostrar_pct_grupo_item=True,
    mostrar_pct_total_item=True,
    casas_decimais_pct=1,
    pad_tiling=3,
    esconder_texto_quando_nao_couber=True,
    tamanho_minimo_texto=10
):
    """
    Gera um treemap agrupado com:
    - universo total opcional
    - grupo em linha compacta
    - itens com múltiplas linhas
    """
    if cor is None:
        cor = coluna_valor

    if titulo_colorbar is None:
        titulo_colorbar = cor

    df_plot = preparar_df_treemap(
        df=df,
        coluna_grupo=coluna_grupo,
        coluna_item=coluna_item,
        coluna_valor=coluna_valor,
        largura_quebra_item=largura_quebra_item,
        nome_coluna_label_item="item_label",
        nome_coluna_label_grupo="label_grupo",
        nome_coluna_label_universo="label_universo",
        abreviar_nome_grupo=abreviar_nome_grupo,
        limite_nome_grupo=limite_nome_grupo,
        incluir_universo=incluir_universo,
        texto_universo=texto_universo
    )

    linhas_item = ["%{label}"]

    if mostrar_valor_item:
        linhas_item.append("%{value:,.0f}")

    if mostrar_pct_grupo_item:
        linhas_item.append(f"%{{percentParent:.{casas_decimais_pct}%}} do grupo")

    if mostrar_pct_total_item:
        linhas_item.append(f"%{{percentRoot:.{casas_decimais_pct}%}} do total")

    texttemplate = "<br>".join(linhas_item)

    if incluir_universo:
        path_treemap = ["label_universo", "label_grupo", "item_label"]
    else:
        path_treemap = ["label_grupo", "item_label"]

    fig = px.treemap(
        df_plot,
        path=path_treemap,
        values=coluna_valor,
        color=cor,
        color_continuous_scale=escala_cor,
        title=titulo
    )

    fig.update_traces(
        texttemplate=texttemplate,
        textfont=dict(size=tamanho_fonte_texto),
        textposition="middle center",
        marker=dict(line=dict(width=1, color="white")),
        tiling=dict(pad=pad_tiling),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Valor: %{value:,.0f}<br>"
            "% do pai: %{percentParent:.1%}<br>"
            "% do total: %{percentRoot:.1%}<extra></extra>"
        )
    )

    fig.update_layout(
        width=largura,
        height=altura,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(t=110, l=30, r=30, b=30),
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        font=dict(size=tamanho_fonte_geral)
    )

    if esconder_texto_quando_nao_couber:
        fig.update_layout(
            uniformtext=dict(
                minsize=tamanho_minimo_texto,
                mode="hide"
            )
        )

    if subtitulo:
        fig.add_annotation(
            x=0.5,
            y=1.04,
            xref="paper",
            yref="paper",
            text=subtitulo,
            showarrow=False,
            xanchor="center",
            font=dict(size=13, color="gray")
        )

    fig.add_annotation(
        x=0.99,
        y=1.08,
        xref="paper",
        yref="paper",
        text="% do grupo = % dentro do grupo pai<br>% do total = % em relação ao total",
        showarrow=False,
        xanchor="right",
        yanchor="bottom",
        align="right",
        font=dict(size=12, color="gray"),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgray",
        borderwidth=1,
        borderpad=4
    )

    if mostrar_colorbar:
        fig.update_layout(
            coloraxis_colorbar=dict(title=titulo_colorbar)
        )
    else:
        fig.update_layout(coloraxis_showscale=False)

    return fig, df_plot

# Dados de demostração

In [56]:
# =========================================
# 1) Geração de dados fake no contexto CID-10
# =========================================
def gerar_dados_fake_cid10(n=3000, seed=42):
    """
    Gera dados fake de atendimentos por grupo CID e CID.

    Cada linha representa 1 atendimento, permitindo agregação posterior.

    Parâmetros
    ----------
    n : int
        Número de registros a gerar.
    seed : int
        Semente para reprodutibilidade.

    Retorna
    -------
    pd.DataFrame
        DataFrame com colunas:
        - cid_grupo
        - cid
        - cid_nome
        - quantidade
    """
    import numpy as np
    import pandas as pd

    np.random.seed(seed)

    cid_grupo = {
        "Doenças do Aparelho Circulatório": [
            {"codigo": "I10", "nome": "Hipertensão essencial (primária)"},
            {"codigo": "I20", "nome": "Angina pectoris"},
            {"codigo": "I50", "nome": "Insuficiência cardíaca"}
        ],
        "Doenças Endócrinas": [
            {"codigo": "E10", "nome": "Diabetes mellitus tipo 1"},
            {"codigo": "E11", "nome": "Diabetes mellitus tipo 2"},
            {"codigo": "E66", "nome": "Obesidade"}
        ],
        "Doenças Respiratórias": [
            {"codigo": "J00", "nome": "Nasofaringite aguda (resfriado comum)"},
            {"codigo": "J18", "nome": "Pneumonia não especificada"},
            {"codigo": "J45", "nome": "Asma"}
        ],
        "Transtornos Mentais": [
            {"codigo": "F10", "nome": "Transtornos por uso de álcool"},
            {"codigo": "F32", "nome": "Episódio depressivo"},
            {"codigo": "F41", "nome": "Transtornos de ansiedade"}
        ],
        "Doenças Osteomusculares": [
            {"codigo": "M17", "nome": "Gonartrose (artrose do joelho)"},
            {"codigo": "M54", "nome": "Dorsalgia"},
            {"codigo": "M79", "nome": "Outros transtornos dos tecidos moles"}
        ]
    }

    nomes_grupos = list(cid_grupo.keys())

    # Distribuição desigual para simular volumes diferentes entre grupos
    probabilidades_grupo = [0.25, 0.20, 0.20, 0.20, 0.15]

    grupos_sorteados = np.random.choice(
        nomes_grupos,
        size=n,
        p=probabilidades_grupo
    )

    cids_sorteados = [
        cid_grupo[grupo][np.random.randint(len(cid_grupo[grupo]))]
        for grupo in grupos_sorteados
    ]

    df_fake = pd.DataFrame({
        "cid_grupo": grupos_sorteados,
        "cid": [item["codigo"] for item in cids_sorteados],
        "cid_nome": [item["nome"] for item in cids_sorteados],
        "quantidade": 1
    })

    return df_fake


In [57]:
# Gera os dados fake
df_fake = gerar_dados_fake_cid10(n=3000, seed=42)

df_agrupado = (
    df_fake.groupby(["cid_grupo", "cid"], as_index=False)
    .agg(quantidade=("quantidade", "sum"))
)

In [58]:
df_agrupado.head(3)

,cid_grupo,cid,quantidade
0,Doenças Endócrinas,E10,182
1,Doenças Endócrinas,E11,191
2,Doenças Endócrinas,E66,201


# Plotagem

In [59]:
fig, df_plot = grafico_treemap_agrupado(
    df=df_agrupado,
    coluna_grupo="cid_grupo",
    coluna_item="cid",
    coluna_valor="quantidade",
    titulo="Distribuição de Atendimentos por CID-10",
    subtitulo="Cada bloco mostra quantidade, % dentro do grupo e % do total.",
    cor="quantidade",
    titulo_colorbar="Quantidade",
    largura=1500,
    altura=900,
    largura_quebra_item=18,
    abreviar_nome_grupo=True,
    limite_nome_grupo=24,
    incluir_universo=True,
    texto_universo="Todos os atendimentos",
    tamanho_fonte_texto=14
)

fig.show()